# Round 3 · Models ⚔️ tier C (25 min)

The remaining operator uses one fresh browser per victim, a residential address in the victim's own city, a mainstream TLS stack, and makes no high-risk change in the first half hour. Nothing links the victims. What is left is each account's own past.

**Unsupervised, per account:** `window_features(df, as_of)` builds features from the last 14 days against the account's earlier behaviour (hour-of-day distance from its usual hour, share of events from its usual device/network/country, timezone and locale shifts, order amount vs its median, new identifiers, alerts turned off) and refuses to look ahead. Score the joint deviation with an IsolationForest; an account's score is its worst week.

**Supervised, for contrast:** Rounds 1 and 2 are the positives; a gradient-boosted model learns what *those* look like.

**Deliverable:** a review queue, the top 100 by anomaly score that Round 2 did not already contain. The scorer reports precision at 25/50/100, because that is what an analyst can open.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../tools"))
from atolab import *
import matplotlib.pyplot as plt
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

from sklearn.ensemble import IsolationForest, HistGradientBoostingClassifier
df = dedupe(load_events())
prev = pd.read_csv("../suspects_r2.csv")
known = set(prev.account_id)
W = 14
WEEKS = ["2026-11-02", "2026-11-09", "2026-11-16", "2026-11-23", "2026-11-30", "2026-12-07", "2026-12-14", "2026-12-21", "2026-12-29"]

## 3.1 Features, week by week (about a minute)

In [ ]:
frames = []
for as_of in WEEKS:
    X = window_features(df, as_of=as_of, windows=(W,))
    X = X[(X.base_events.fillna(0) >= 8) & (X[f"w{W}_events"] > 0)].copy()   # no baseline, no score
    X["as_of"] = as_of
    frames.append(X)
F = pd.concat(frames)
F["amount_ratio"] = np.log1p(F.amount_ratio.fillna(1.0).clip(upper=50)); F["days_since_base"] = np.log1p(F.days_since_base.fillna(0)); F["hour_delta"] = F.hour_delta.fillna(0)
print(F.shape); F.filter(like="w14_").describe().T.head(8)

## 3.2 Deviation features and the isolation forest

Pick the features that describe *change relative to this account*, not raw volume. Then look at your queue's composition with `timeline()` before you submit it: which cohorts does it contain, and what single feature put them there?

In [ ]:
DEVIATION = ["hour_delta", f"w{W}_tz_shift", f"w{W}_lang_shift", f"w{W}_share_base_device", f"w{W}_share_base_asn", f"w{W}_share_base_country",
             f"w{W}_new_ids", f"w{W}_alerts_off", "amount_ratio", f"w{W}_med", f"w{W}_high", f"w{W}_logins", f"w{W}_resume_share", "days_since_base", f"w{W}_devices", f"w{W}_asns"]
iso = IsolationForest(n_estimators=300, random_state=2026).fit(F[DEVIATION].fillna(0))
F["anomaly"] = -iso.score_samples(F[DEVIATION].fillna(0))
per_acc = F.groupby(level=0).anomaly.max().sort_values(ascending=False)
queue = per_acc[~per_acc.index.isin(known)].head(100)
queue.head(10)

In [ ]:
timeline(df, queue.index[0], n=25)

## 3.3 The supervised contrast

Train on what you already found. Submit its queue instead (swap `queue` below) and compare the per-tier numbers.

In [ ]:
agg = F.groupby(level=0)[DEVIATION].max()
y = agg.index.isin(known).astype(int)
clf = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=2026).fit(agg, y)
p_sup = pd.Series(clf.predict_proba(agg)[:, 1], index=agg.index)
sup_queue = p_sup[~p_sup.index.isin(known)].sort_values(ascending=False).head(100)

## 3.4 Submit (cumulative) and score

In [ ]:
out = pd.concat([pd.DataFrame({"account_id": prev.account_id, "score": 10 + prev.score.fillna(0), "first_event_id": prev.first_event_id}),
                 pd.DataFrame({"account_id": queue.index, "score": queue.values, "first_event_id": None})]).drop_duplicates("account_id")
save_suspects(out[["account_id"]], "../suspects_r3.csv", scores=out.score, first_event=out.first_event_id)
score_now("../suspects_r3.csv", "--prev", "../suspects_r2.csv")

## 3.5 Before the reveal

- The queue is 100 accounts. How many are takeovers, and what would the owners of the rest experience if you acted on it automatically?
- Which feature would you add? (The attacker reads this notebook too.)
- Bonus: a temporal split, train on days 1–60, evaluate on 61–90.

Then `04_reveal.ipynb`, or `python tools/checkpoint.py --reveal`.